<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/05_TF_IDF_Say%C4%B1sal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Duygu Analizi Model Eğitimi ve Karşılaştırma

Bu defterde, Amazon Datset Reviews 2023 üzerindne aldığımız Electronic kategorisi üzerinden metin verilerine ek olarak `char_count` , `exclamation_count`, `question_count`,`uppercase_count`, `avg_word_len`, `verified_purchase` sütunlarınıda kullanarak  duygu analizi yapıyoruz.Önceki defterlerde örneklem çıkarılıp ,veriseti üstünde analiz yapılıp ,ön işleme adımları yapılmıştır.04A_Text_Title adlı defterde ise sadece text ve title sütunları kullanılarak model eğitimine verilmiştir.Bu defterde ise metin verilerimize ek olarak yularıda belirttiğimiz sayısal sütunlarımızda ,TF-IDF ile vektörüze edilen  metin verilerimizle beraber model eğitimine verilmiştir.




### Yapılan Adımlar:

1.  **Veri Yükleme ve Ön İşleme**: `amazon_reviews_preprocessed_FINAL.csv` adlı veri seti yüklenmiş, 'clean_text' ve 'clean_title' sütunlarındaki eksik değerler doldurulmuş ve boş metinler filtrelenmiştir.
2.  **Veri Bölme**: Veri seti, eğitim, doğrulama (validation) ve test setlerine stratifiye edilmiş şekilde ayrılmıştır.Veri seti test seti %15 validation seti %15 train seti %70 olacak şekilde ayrılmıştır.
3.  **TF-IDF Vektörizasyonu**: Hem başlık (title) hem de metin (text) sütunları için TF-IDF (Term Frequency-Inverse Document Frequency) vektörizasyonu uygulanmış, ardından bu vektörler birleştirilerek tek bir özellik matrisi oluşturulmuştur.
4.  **Model Eğitimi ve Değerlendirme**: Logistic Regression,  Linear SVM, XGBoost ve LightGBM gibi farklı sınıflandırma modelleri, birleştirilmiş TF-IDF matrisi üzerinde eğitilmiş ve her bir modelin performansı doğrulama ve test setlerinde ayrı ayrı değerlendirilmiştir.
5.  **Model Karşılaştırması**: Eğitilen modellerin performans metrikleri (Accuracy, Precision, Recall, F1-Score, ROC-AUC) bir karşılaştırma tablosunda sunulmuş ve görsellerle desteklenmiştir. ROC eğrileri de karşılaştırma için çizilmiştir.
6.  **Özellik Önem Analizi**: Logistic Regression modeli özelinde, en etkili pozitif ve negatif kelimeler belirlenerek görselleştirilmiştir.
7.  **Hata Analizi**: En iyi performans gösteren modelin yanlış sınıflandırdığı örnekler (False Positive ve False Negative) incelenmiştir.
8.  **Model Kaydetme ve Yeni Tahminler**: En iyi performansı gösteren model ve TF-IDF vektörleyicileri kaydedilmiş, ayrıca yeni yorumlar için duygu tahmini yapabilen bir fonksiyon örneği sunulmuştur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import time
import joblib

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('Kütüphaneler yüklendi.')

# Veri Seti Okuma
Kullanacağımız csv dosyasının ilk 5 satırı listelenerek csv dosyasının satır ve sütun bilgileri alınmıştır.

In [ ]:
input_path = '/content/drive/MyDrive/Veri_madenciliği/Dataset/amazon_reviews_preprocessed_FINAL.csv'

df = pd.read_csv(input_path)

print(f'Veri seti yüklendi: {df.shape[0]:,} satır, {df.shape[1]} sütun')
print(f'\nSütunlar: {df.columns.tolist()}')

display(df.head())

# Veri Temizliği ve Sayısal Özelliklerin  Hazırlanması
Bu hücrede, metin (TF-IDF) verilerine ek olarak modele beslenecek sayısal özelliklerin ön hazırlığı yapılmıştır.

 **Eksik Veri Yönetimi:** Hem başlık (clean_title) hem de içerik (clean_text) sütunlarındaki boş değerler temizlenmiştir. Bilgi kaybını önlemek amacıyla, sadece her iki alanı da tamamen boş olan satırlar veri setinden çıkarılmıştır.

**Veri Tipi Dönüşümü:** Modelin matematiksel olarak işleyebilmesi için mantıksal yapıdaki "Onaylı Satın Alma" (verified_purchase) sütunu (True/False), ikili (binary) formata (1/0) dönüştürülmüştür.

**Sayısal Özellik Seçimi:** Modelin metnin duygusunu sadece kelimelerden değil, yazım biçiminden de (örn: büyük harf kullanımı, ünlem sayısı, yorum uzunluğu) öğrenebilmesi için 6 farklı istatistiksel özellik belirlenmiş ve bu verilerin genel dağılım tablosu (describe) incelenmek üzere ekrana yazdırılmıştır.

In [ ]:

print(' ETİKET DAĞILIMI \n')
print(df['label'].value_counts())

# clean_text ve clean_title'ı temizle
df['clean_text'] = df['clean_text'].fillna('').astype(str)
df['clean_title'] = df['clean_title'].fillna('').astype(str)
df = df[(df['clean_text'].str.strip() != '') | (df['clean_title'].str.strip() != '')].reset_index(drop=True)

# verified_purchase'ı 0/1'e çevir
if df['verified_purchase'].dtype == bool:
    df['verified_purchase'] = df['verified_purchase'].astype(int)
elif df['verified_purchase'].dtype == object:
    df['verified_purchase'] = df['verified_purchase'].map({'True': 1, 'False': 0, True: 1, False: 0}).fillna(0).astype(int)

# Sayısal sütunlar
sayisal_kolonlar = [
    'char_count', 'exclamation_count', 'question_count',
    'uppercase_count', 'avg_word_len', 'verified_purchase'
]

print(f'\nKullanılacak özellikler:')
print(f'  - clean_title (TF-IDF)')
print(f'  - clean_text  (TF-IDF)')
print(f'  - 6 sayısal feature: {sayisal_kolonlar}')

print(f'\nSayısal feature\'ların özeti:')
display(df[sayisal_kolonlar].describe().round(2))

# Veri Setinin Bölünmesi (Train-Val-Test Split)
Modelin ezberlemesini önlemek ve doğru değerlendirilmesini sağlamak amacıyla veri seti üç alt kümeye ayrılmıştır. Veri bütünlüğünü korumak için bölme işlemi doğrudan veriler yerine DataFrame indeksleri karıştırılarak yapılmıştır. Böylece her bir yoruma ait başlık, içerik, sayısal özellikler ve hedef etiket (0/1) arasındaki hizalama kusursuz şekilde korunmuştur.

Aşamalar:
Test Setinin Ayrılması: Tüm indekslerin %15'i, sınıf dağılımları korunarak (stratify) test seti olarak ayrılmıştır.
Eğitim ve Doğrulama (Validation) Setlerinin Ayrılması: Geriye kalan %85'lik geçici veri üzerinden ikinci bir bölme işlemi yapılmıştır. Toplam veri setinde %15'lik bir paya ulaşmak amacıyla, bu geçici setin %17.65'i Doğrulama (Validation) seti olarak belirlenmiş, geri kalan veriler ise Eğitim (Train) seti olarak bırakılmıştır. Model uyumluluğu için sayısal özellikler float (ondalıklı) formata çevrilmiştir.

Dağılım: %70 Eğitim, %15 Doğrulama ve %15 Test şeklindedir.

In [ ]:
# Hedef değişken
y = df['label']

# Sayısal sütunlar
sayisal_kolonlar = [
    'char_count', 'exclamation_count', 'question_count',
    'uppercase_count', 'avg_word_len', 'verified_purchase'
]

# === SPLIT (İndeks bazlı) ===
indeksler = df.index

idx_temp, idx_test = train_test_split(
    indeksler, test_size=0.15, stratify=y, random_state=42
)
idx_train, idx_val = train_test_split(
    idx_temp, test_size=0.1765, stratify=y.loc[idx_temp], random_state=42
)

# Title
X_train_title = df.loc[idx_train, 'clean_title']
X_val_title   = df.loc[idx_val,   'clean_title']
X_test_title  = df.loc[idx_test,  'clean_title']

# Text
X_train_text = df.loc[idx_train, 'clean_text']
X_val_text   = df.loc[idx_val,   'clean_text']
X_test_text  = df.loc[idx_test,  'clean_text']

# Sayısal feature'lar
X_train_num = df.loc[idx_train, sayisal_kolonlar].astype(float)
X_val_num   = df.loc[idx_val,   sayisal_kolonlar].astype(float)
X_test_num  = df.loc[idx_test,  sayisal_kolonlar].astype(float)

# Hedef
y_train = y.loc[idx_train]
y_val   = y.loc[idx_val]
y_test  = y.loc[idx_test]

toplam = len(df)
print('*** VERİ BÖLÜNDÜ ***\n')
print(f'Train      : {len(idx_train):,} ({len(idx_train)/toplam*100:.1f}%)')
print(f'Validation : {len(idx_val):,} ({len(idx_val)/toplam*100:.1f}%)')
print(f'Test       : {len(idx_test):,} ({len(idx_test)/toplam*100:.1f}%)')

print('\nTrain etiket dağılımı:')
print(y_train.value_counts(normalize=True).round(3))

# TF-IDF ve Sayısal Özelliklerin Birleştirilmesi
Modelin farklı türdeki verileri aynı ölçekte anlayabilmesi için metin ve sayısal özellikler işlenerek tek bir matriste birleştirilmiştir.


**Metinlerin Vektörizasyonu (TF-IDF):** Başlık (Title) verilerinden en önemli 5.000, içerik (Text) verilerinden ise 20.000 kelime/öbek TF-IDF yöntemiyle sayısallaştırılmıştır. Sızıntıyı önlemek için sözlük sadece Eğitim setinden öğrenilmiştir.


**Sayısal Özelliklerin Ölçeklenmesi:** TF-IDF skorları (0-1 arası) ile metin uzunluğu gibi büyük sayıların modelde dengesizlik yaratmaması için, sayısal özellikler MinMaxScaler kullanılarak 0-1 aralığına sıkıştırılmıştır. Ölçekleme kuralları sadece Eğitim seti kullanılarak belirlenmiştir.


**Matrislerin Birleştirilmesi (hstack):** Elde edilen 5.000 sütunluk başlık, 20.000 sütunluk içerik ve 6 sütunluk sayısal veri matrisleri yan yana birleştirilmiş (toplam 25.006 özellik) ve bellek optimizasyonu için CSR formatında sıkıştırılmıştır.

In [ ]:
print('TF-IDF + Sayısal Feature vektörizasyonu başladı...\n')

#  TITLE TF-IDF
tfidf_title = TfidfVectorizer(
    ngram_range=(1, 3), min_df=10, max_df=0.95,
    max_features=5000, sublinear_tf=True
)
X_train_title_tfidf = tfidf_title.fit_transform(X_train_title)
X_val_title_tfidf   = tfidf_title.transform(X_val_title)
X_test_title_tfidf  = tfidf_title.transform(X_test_title)
print(f'Title TF-IDF shape: {X_train_title_tfidf.shape}')

#  TEXT TF-IDF
tfidf_text = TfidfVectorizer(
    ngram_range=(1, 3), min_df=10, max_df=0.95,
    max_features=20000, sublinear_tf=True
)
X_train_text_tfidf = tfidf_text.fit_transform(X_train_text)
X_val_text_tfidf   = tfidf_text.transform(X_val_text)
X_test_text_tfidf  = tfidf_text.transform(X_test_text)
print(f'Text TF-IDF shape : {X_train_text_tfidf.shape}')

# SAYISAL FEATURE'LAR — MinMaxScaler (sadece train'e fit)
scaler = MinMaxScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_val_num_scaled   = scaler.transform(X_val_num)
X_test_num_scaled  = scaler.transform(X_test_num)

print(f'Sayısal feature shape: {X_train_num_scaled.shape}')
print(f'\nÖlçekleme sonrası min/max:')
for i, col in enumerate(sayisal_kolonlar):
    print(f'  {col:20s}: [{X_train_num_scaled[:, i].min():.3f}, {X_train_num_scaled[:, i].max():.3f}]')

#  HSTACK: Title + Text + Sayısal
X_train_combined = hstack([
    X_train_title_tfidf,
    X_train_text_tfidf,
    csr_matrix(X_train_num_scaled)
]).tocsr()

X_val_combined = hstack([
    X_val_title_tfidf,
    X_val_text_tfidf,
    csr_matrix(X_val_num_scaled)
]).tocsr()

X_test_combined = hstack([
    X_test_title_tfidf,
    X_test_text_tfidf,
    csr_matrix(X_test_num_scaled)
]).tocsr()

print(f'\n BİRLEŞİK MATRİS ')
print(f'Train: {X_train_combined.shape}')
print(f'Val  : {X_val_combined.shape}')
print(f'Test : {X_test_combined.shape}')
print(f'Toplam feature: {X_train_combined.shape[1]:,}')
print(f'  - Title TF-IDF : {X_train_title_tfidf.shape[1]:,}')
print(f'  - Text TF-IDF  : {X_train_text_tfidf.shape[1]:,}')
print(f'  - Sayısal      : {X_train_num_scaled.shape[1]}')

In [ ]:
# Title ve Text için en yüksek IDF skorlu (en nadir) terimleri ayrı ayrı görelim
print(' TITLE - En Yüksek IDF (En Nadir) 15 Terim ')
title_idf = pd.DataFrame({
    'term': tfidf_title.get_feature_names_out(),
    'idf': tfidf_title.idf_
}).sort_values('idf', ascending=False)
display(title_idf.head(15))

print('\n*** TEXT - En Yüksek IDF (En Nadir) 15 Terim ***')
text_idf = pd.DataFrame({
    'term': tfidf_text.get_feature_names_out(),
    'idf': tfidf_text.idf_
}).sort_values('idf', ascending=False)
display(text_idf.head(15))

# Trigram örnekleri (gerçekten trigram yakalanıyor mu görelim)
print('\n*** TEXT - Trigram (3-gram) Örnekleri ***')
trigrams = [t for t in tfidf_text.get_feature_names_out() if len(t.split()) == 3]
print(f'Toplam trigram sayısı: {len(trigrams):,}')
print('İlk 20 trigram örneği:')
for tg in trigrams[:20]:
    print(f'  - {tg}')

# Model Eğitim ve Değerlendirme Fonksiyonu

Farklı makine öğrenmesi modellerini denerken kod tekrarını önlemek ve süreci standartlaştırmak amacıyla bu otomatik test fonksiyonu hazırlanmıştır.

Fonksiyona gönderilen her model için sırasıyla şu işlemler gerçekleştirilir:

**Eğitim (Training) ve Süre Ölçümü:** Model, Eğitim setiyle eğitilir ve bu işlemin hız/maliyet analizi için kaç saniye sürdüğü kaydedilir.


**Performans Metriklerinin Hesaplanması:** Modelin Doğrulama (Validation) ve Test setlerindeki başarısı; Accuracy (Doğruluk), Precision (Kesinlik), Recall (Duyarlılık), F1-Score ve ROC-AUC metrikleriyle ayrı ayrı ölçülür.

**Raporlama ve Görselleştirme:** Sınıflar bazında (Pozitif/Negatif) detaylı hata analizi yapabilmek için Sınıflandırma Raporu (Classification Report) yazdırılır ve modelin nerelerde yanıldığını görmek için Karmaşıklık Matrisi (Confusion Matrix) ısı haritası olarak çizdirilir.


**Sonuçların Kaydedilmesi:** Elde edilen tüm metrikler ve süre bilgileri paketlenerek (dictionary formatında) dışarı aktarılır. Bu sayede projenin sonunda denenen tüm modeller tek bir veri çerçevesinde (DataFrame) kolayca karşılaştırılabilecektir.

In [ ]:
def model_egit_ve_degerlendir(model, model_adi, X_train, y_train, X_val, y_val, X_test, y_test):

    print(f'\n{"="*70}')
    print(f' MODEL: {model_adi}')
    print(f'{"="*70}')

    #  Eğitim
    start = time.time()
    model.fit(X_train, y_train)
    egitim_suresi = time.time() - start
    print(f'Eğitim süresi: {egitim_suresi:.2f} saniye')

    #  Tahminler (val + test)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    #  Olasılık skorları (ROC için)
    if hasattr(model, 'predict_proba'):
        y_val_score = model.predict_proba(X_val)[:, 1]
        y_test_score = model.predict_proba(X_test)[:, 1]
    else:
        y_val_score = model.decision_function(X_val)
        y_test_score = model.decision_function(X_test)

    #  VALIDATION SETİ METRİKLERİ
    val_acc = accuracy_score(y_val, y_val_pred)
    val_prec = precision_score(y_val, y_val_pred)
    val_rec = recall_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    val_auc = roc_auc_score(y_val, y_val_score)

    print(f'\n VALIDATION SETİ METRİKLERİ ')
    print(f'Accuracy : {val_acc:.4f}')
    print(f'Precision: {val_prec:.4f}')
    print(f'Recall   : {val_rec:.4f}')
    print(f'F1-Score : {val_f1:.4f}')
    print(f'ROC-AUC  : {val_auc:.4f}')

    # TEST SETİ METRİKLERİ
    test_acc = accuracy_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred)
    test_rec = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    test_auc = roc_auc_score(y_test, y_test_score)

    print(f'\n TEST SETİ METRİKLERİ ')
    print(f'Accuracy : {test_acc:.4f}')
    print(f'Precision: {test_prec:.4f}')
    print(f'Recall   : {test_rec:.4f}')
    print(f'F1-Score : {test_f1:.4f}')
    print(f'ROC-AUC  : {test_auc:.4f}')

    #  Test seti için classification report
    print(f'\n*** TEST SETİ SINIFLANDIRMA RAPORU ***')
    print(classification_report(y_test, y_test_pred, target_names=['Negatif (0)', 'Pozitif (1)']))

    #  Test seti için confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['Negatif', 'Pozitif'],
        yticklabels=['Negatif', 'Pozitif']
    )
    plt.title(f'{model_adi} - Test Seti Confusion Matrix')
    plt.ylabel('Gerçek Etiket')
    plt.xlabel('Tahmin Edilen Etiket')
    plt.tight_layout()
    plt.show()

    return {
        'model': model,
        'model_adi': model_adi,
        # Validation metrikleri
        'val_accuracy': val_acc,
        'val_precision': val_prec,
        'val_recall': val_rec,
        'val_f1': val_f1,
        'val_roc_auc': val_auc,
        # Test metrikleri
        'test_accuracy': test_acc,
        'test_precision': test_prec,
        'test_recall': test_rec,
        'test_f1': test_f1,
        'test_roc_auc': test_auc,
        # Zaman + tahminler
        'egitim_suresi': egitim_suresi,
        'y_test_pred': y_test_pred,
        'y_test_score': y_test_score
    }


tum_sonuclar = []

 Logistic Regresyon ile Model Eğitimi

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='liblinear',
    random_state=42
)

sonuc_lr = model_egit_ve_degerlendir(
    lr_model, 'Logistic Regression',
    X_train_combined, y_train,
    X_val_combined, y_val,
    X_test_combined, y_test
)
tum_sonuclar.append(sonuc_lr)

Naive Bayes ile Model Eğitimi

In [ ]:
nb_model = MultinomialNB(alpha=1.0)

sonuc_nb = model_egit_ve_degerlendir(
    nb_model, 'Multinomial Naive Bayes',
    X_train_combined, y_train,
    X_val_combined, y_val,
    X_test_combined, y_test
)
tum_sonuclar.append(sonuc_nb)

Lİnear SVC ile Model Eğitimi

In [ ]:
svm_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

sonuc_svm = model_egit_ve_degerlendir(
    svm_model, 'Linear SVM',
    X_train_combined, y_train,
    X_val_combined, y_val,
    X_test_combined, y_test
)
tum_sonuclar.append(sonuc_svm)

XGBOOST ile Model Eğitimi

In [ ]:
# XGBoost yüklemesi
!pip install -q xgboost

import xgboost as xgb
print(f'XGBoost versiyonu: {xgb.__version__}')

# GPU yerine CPU kullan (TF-IDF sparse veride GPU bug riski)
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',     # GPU yerine CPU hist
    device='cpu',           # ← GPU'yu kapat
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

sonuc_xgb = model_egit_ve_degerlendir(
    xgb_model, 'XGBoost',
    X_train_combined, y_train,
    X_val_combined, y_val,
    X_test_combined, y_test
)
tum_sonuclar.append(sonuc_xgb)

LİghtGBM ile Model Eğitimi

In [ ]:
!pip install -q lightgbm

import lightgbm as lgb
print(f'LightGBM versiyonu: {lgb.__version__}')

lgb_model = lgb.LGBMClassifier(
    n_estimators=300, num_leaves=63, learning_rate=0.1,
    objective='binary', metric='binary_logloss',
    random_state=42, n_jobs=-1, verbose=-1
)

sonuc_lgb = model_egit_ve_degerlendir(
    lgb_model, 'LightGBM',
    X_train_combined, y_train,
    X_val_combined, y_val,
    X_test_combined, y_test
)
tum_sonuclar.append(sonuc_lgb)

# Model Karşılaştırma
Eğitilen 5 model bir tabloda karşılaştırma işlemi yapılır.

In [ ]:
# Karşılaştırma DataFrame'i — hem Validation hem Test skorlarını göster
karsilastirma_df = pd.DataFrame([
    {
        'Model': s['model_adi'],
        'Val Accuracy': round(s['val_accuracy'], 4),
        'Val F1': round(s['val_f1'], 4),
        'Val ROC-AUC': round(s['val_roc_auc'], 4),
        'Test Accuracy': round(s['test_accuracy'], 4),
        'Test Precision': round(s['test_precision'], 4),
        'Test Recall': round(s['test_recall'], 4),
        'Test F1': round(s['test_f1'], 4),
        'Test ROC-AUC': round(s['test_roc_auc'], 4),
        'Eğitim Süresi (sn)': round(s['egitim_suresi'], 2)
    }
    for s in tum_sonuclar
]).sort_values('Test F1', ascending=False).reset_index(drop=True)

print('MODEL KARŞILAŞTIRMA TABLOSU \n')
display(karsilastirma_df)

# CSV olarak kaydet
output_dir = '/content/drive/MyDrive/Veri_madenciliği/Sonuclar/'
os.makedirs(output_dir, exist_ok=True)
karsilastirma_df.to_csv(output_dir + '04B_hybrid_karsilastirma.csv', index=False, encoding='utf-8-sig')
print(f'\nTablo kaydedildi: {output_dir}04B_hybrid_karsilastirma.csv')

# Model Performanslarının Görselleştirilmesi ve Karşılaştırması
Bu hücrede, eğitilen tüm modellerin performans metrikleri (Accuracy, F1-Score, ROC-AUC) yan yana çubuk grafikler (bar chart) halinde görselleştirilmiştir.

**İkili Görünüm (Validation vs Test):** Sol tarafta modellerin Validation (Doğrulama) skorları, sağ tarafta ise Test skorları konumlandırılmıştır. Bu dizilim, modellerin hiç görmediği verilerdeki başarısını kıyaslamanın yanı sıra, aşırı öğrenme (overfitting) yapıp yapmadıklarını tek bakışta analiz etmemizi sağlar.

**Metriklerin Kıyaslanması:** Her model için Accuracy, F1 ve ROC-AUC değerleri farklı renkli çubuklarla gösterilerek modellerin hem genel doğruluğu hem de sınıflandırma kalitesi karşılaştırılmıştır.

**Dışa Aktarma:** Oluşturulan nihai grafik, proje raporu veya sunumlarda kullanılmaya hazır olacak şekilde yüksek çözünürlüklü bir PNG dosyası olarak kaydedilir.

In [ ]:
# Validation vs Test karşılaştırma grafiği
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrikler = ['Accuracy', 'F1', 'ROC-AUC']
x = np.arange(len(karsilastirma_df))
width = 0.25

# Sol grafik: Validation skorları
for i, m in enumerate(metrikler):
    axes[0].bar(x + i * width, karsilastirma_df[f'Val {m}'], width, label=m)
axes[0].set_title('Validation Seti Skorları', fontsize=13, fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(karsilastirma_df['Model'], rotation=15)
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', linestyle='--', alpha=0.6)

# Sağ grafik: Test skorları
for i, m in enumerate(metrikler):
    axes[1].bar(x + i * width, karsilastirma_df[f'Test {m}'], width, label=m)
axes[1].set_title('Test Seti Skorları', fontsize=13, fontweight='bold')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(karsilastirma_df['Model'], rotation=15)
axes[1].set_ylim(0, 1.05)
axes[1].legend(loc='lower right')
axes[1].grid(axis='y', linestyle='--', alpha=0.6)

plt.suptitle('TF-IDF Tabanlı Model Karşılaştırması', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(output_dir + 'tfidf_model_karsilastirma.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Grafik kaydedildi: {output_dir}tfidf_model_karsilastirma.png')

# ROC Eğrileri (ROC Curve) Karşılaştırması
Bu hücrede, eğitilen tüm modellerin sınıflandırma performansları ROC Eğrisi (Receiver Operating Characteristic) kullanılarak tek bir grafik üzerinde karşılaştırılmıştır.

**Eğrinin Anlamı:** ROC eğrisi, modelin Doğru Pozitif Oranı (TPR) ile Yanlış Pozitif Oranı (FPR) arasındaki dengeyi gösterir. Bir eğri sol üst köşeye ne kadar yakınsa, modelin ayırt edicilik gücü o kadar yüksektir.

**AUC Skoru:** Lejantta belirtilen AUC (Area Under Curve) değeri, eğrinin altında kalan alanı temsil eder. Rastgele tahmin yapan bir modelin AUC skoru 0.5'tir (siyah kesik çizgi). Modellerimizin AUC skorunun 1.0'a ne kadar yakın olduğu, başarısını doğrudan yansıtır.

Tüm modellerin sonuçları üst üste bindirilerek, test verisi üzerinde en iyi ayrımı yapan algoritma görsel olarak tespit edilmiş ve grafik raporlama için dışa aktarılmıştır.

In [ ]:
plt.figure(figsize=(9, 7))

for sonuc in tum_sonuclar:
    fpr, tpr, _ = roc_curve(y_test, sonuc['y_test_score'])
    plt.plot(fpr, tpr, lw=2,
             label=f"{sonuc['model_adi']} (AUC = {sonuc['test_roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Rastgele Tahmin (AUC = 0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Eğrileri Karşılaştırması (Test Seti)', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir + '04B_hybrid_roc.png', dpi=200, bbox_inches='tight')
plt.show()

# Modelin Karar Mekanizmasının Analizi
Bu hücrede, Lojistik Regresyon modelinin tahmin yaparken sadece hangi kelimelere değil, aynı zamanda hangi sayısal özelliklere (metadata) odaklandığı (Feature Importance) incelenmiştir.

Doğrusal modeller, eğitim sırasında her bir özelliğe (kelimeye veya istatistiksel değere) pozitif veya negatif bir matematiksel katsayı (ağırlık) atar. Pozitif katsayısı en yüksek olan özellikler modeli "Olumlu (1)" sınıfına, en düşük (negatif) olanlar ise "Olumsuz (0)" sınıfına yönlendirir.

Kod içerisinde Başlık (Title) sözlüğü, Metin (Text) sözlüğü ve 6 adet sayısal özelliğin isimleri uç uca eklenmiş (np.concatenate), modelin katsayıları (coef_) ile eşleştirilmiş ve modelin kararlarını en çok etkileyen Top 20 Pozitif ve Top 20 Negatif özellik (kelime veya istatistik) çıkarılarak modelin şeffaflığı (explainability) test edilmiştir.

In [ ]:
def en_etkili_kelimeler(model, feature_names_list, top_n=20):

    feature_names = feature_names_list # Use the passed list directly
    coefs = model.coef_[0]

    if len(coefs) != len(feature_names):
        print(f"Warning: Number of coefficients ({len(coefs)}) does not match number of feature names ({len(feature_names)}). Data mismatch!")


    df_coefs = pd.DataFrame({'kelime': feature_names, 'katsayi': coefs})

    en_pozitif = df_coefs.nlargest(top_n, 'katsayi').reset_index(drop=True)
    en_negatif = df_coefs.nsmallest(top_n, 'katsayi').reset_index(drop=True)

    return en_pozitif, en_negatif

all_feature_names = (
    tfidf_title.get_feature_names_out().tolist() +
    tfidf_text.get_feature_names_out().tolist() +
    sayisal_kolonlar
)

# Logistic Regression'dan çıkar

pos_words, neg_words = en_etkili_kelimeler(sonuc_lr['model'], all_feature_names, top_n=20)

print(' EN POZİTİF 20 KELİME (Logistic Regression) ')
display(pos_words)

print('\n EN NEGATİF 20 KELİME (Logistic Regression) ')
display(neg_words)

In [ ]:
# Görselleştirme: yan yana iki bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

sns.barplot(data=pos_words, y='kelime', x='katsayi', color='seagreen', ax=axes[0])
axes[0].set_title('En POZİTİF 20 Kelime', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Katsayı (Pozitif Etki)')
axes[0].set_ylabel('')

sns.barplot(data=neg_words, y='kelime', x='katsayi', color='indianred', ax=axes[1])
axes[1].set_title('En NEGATİF 20 Kelime', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Katsayı (Negatif Etki)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(output_dir + '04B_en_etkili_kelimeler.png', dpi=200, bbox_inches='tight')
plt.show()

# En İyi Modelin Seçimi ve Hata Analizi (Error Analysis)
Modellerin genel başarı metrikleri hesaplandıktan sonra, iyileştirme stratejileri geliştirebilmek için detaylı bir hata analizine geçilmiştir.

**Model Seçimi:** Tüm algoritmalar arasından Test setindeki F1-Skoru en yüksek olan model nihai model (şampiyon) olarak belirlenmiştir.

**Tahmin ve Gerçeklik Eşleşmesi:** Modelin karar mekanizmasını insan gözüyle inceleyebilmek adına; modelin matematiksel tahminleri, test setindeki orijinal yorum metinleri (text) ve gerçek etiketlerle (label) tek bir DataFrame üzerinde birleştirilmiştir.

**Hata Sınıflandırması:** Modelin yanıldığı tüm örnekler filtrelenmiş ve iki ana kategoriye ayrılmıştır:

**False Positive (Tip I Hata):** Gerçekte Negatif (0) olup, modelin Pozitif (1) zannettiği durumlar.

**False Negative (Tip II Hata):** Gerçekte Pozitif (1) olup, modelin Negatif (0) zannettiği durumlar.

In [ ]:
# Test F1'i en yüksek olan modeli seç
en_iyi_idx = np.argmax([s['test_f1'] for s in tum_sonuclar])
en_iyi_sonuc = tum_sonuclar[en_iyi_idx]

print(f"En iyi model: {en_iyi_sonuc['model_adi']} (Test F1 = {en_iyi_sonuc['test_f1']:.4f})\n")

# Tahminleri orijinal metinle birleştir
hata_df = pd.DataFrame({
    'metin': X_test_text.values,
    'gercek': y_test.values,
    'tahmin': en_iyi_sonuc['y_test_pred']
})

# Yanlış sınıflandırılanlar
yanlislar = hata_df[hata_df['gercek'] != hata_df['tahmin']].copy()
print(f'Toplam yanlış tahmin sayısı: {len(yanlislar):,} / {len(hata_df):,} ({len(yanlislar)/len(hata_df)*100:.2f}%)')

false_positives = yanlislar[(yanlislar['gercek'] == 0) & (yanlislar['tahmin'] == 1)]
false_negatives = yanlislar[(yanlislar['gercek'] == 1) & (yanlislar['tahmin'] == 0)]

print(f'False Positive (Negatif → Pozitif yanılgı): {len(false_positives):,}')
print(f'False Negative (Pozitif → Negatif yanılgı): {len(false_negatives):,}')

In [ ]:
print(' ÖRNEK FALSE POSITIVE YORUMLAR ')
print('(Aslında NEGATİF ama model POZİTİF sandı)\n')
for i, row in false_positives.head(5).iterrows():
    print(f"- {row['metin'][:200]}...\n")

print('\n*** ÖRNEK FALSE NEGATIVE YORUMLAR ***')
print('(Aslında POZİTİF ama model NEGATİF sandı)\n')
for i, row in false_negatives.head(5).iterrows():
    print(f"- {row['metin'][:200]}...\n")

# Hibrit Model ve Veri Dönüştürücülerin Dışa Aktarılması
Bu hücrede, eğitim süreci tamamlanan "En İyi Hibrit Model" ve veri ön işleme aşamasında kullanılan dönüştürme araçları kalıcı olarak kaydedilmiştir.

Yeni ve daha önce hiç görülmemiş veriler üzerinde doğru tahmin (inference) yapılabilmesi için modelin tek başına kaydedilmesi yeterli değildir. Modelin eğitildiği veri formatına sadık kalmak adına; metinleri sayısallaştıran TF-IDF dönüştürücüleri (tfidf_title ve tfidf_text) ile sayısal (metadata) özellikleri 0-1 aralığına ölçekleyen MinMaxScaler aracı modelin ayrılmaz birer parçası olarak joblib kütüphanesi ile dışa aktarılmıştır. Kaydedilen bu 4 dosya, modeli yeniden eğitime gerek kalmadan deployment (canlıya alma) aşamasında doğrudan kullanılmak üzere hedef dizine aktarılmıştır.

In [ ]:
model_dir = '/content/drive/MyDrive/Veri_madenciliği/Models/'
os.makedirs(model_dir, exist_ok=True)

model_yolu = model_dir + f"best_04B_hybrid_model_{en_iyi_sonuc['model_adi'].replace(' ', '_')}.joblib"
tfidf_title_yolu = model_dir + 'tfidf_title_04B.joblib'
tfidf_text_yolu = model_dir + 'tfidf_text_04B.joblib'
scaler_yolu = model_dir + 'minmax_scaler_04B.joblib'

joblib.dump(en_iyi_sonuc['model'], model_yolu)
joblib.dump(tfidf_title, tfidf_title_yolu)
joblib.dump(tfidf_text, tfidf_text_yolu)
joblib.dump(scaler, scaler_yolu)

print(f'Model kaydedildi          : {model_yolu}')
print(f'Title vectorizer          : {tfidf_title_yolu}')
print(f'Text vectorizer           : {tfidf_text_yolu}')
print(f'MinMax scaler             : {scaler_yolu}')